# The dashboard's data, from the MCP server

This notebook reads the same tools Claude and the Gemini analyst use, from a running
`qbs` MCP server. The server loads the data once and keeps it fresh; the notebook just
asks for it.

**On the machine with the data** (e.g. the mini PC), start the server:

```bash
# make a token once:  python -c "import secrets; print(secrets.token_urlsafe(32))"
QBS_MCP_TOKEN=<token> python -m qbs.agent.mcp_server --http --host 0.0.0.0
```

**Here**, set the URL and token below, or put `QBS_MCP_URL` / `QBS_MCP_TOKEN` in the repo's
`.env`. With neither set, `connect()` starts a private server locally from this checkout's
cache instead.

Needs `pip install -r requirements-mcp.txt` (and `requirements-agent.txt` for the last
section).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # the repo root, when run from notebooks/

from qbs.agent.mcp_client import connect

URL = None      # e.g. "http://minipc.local:8765/mcp"; None -> QBS_MCP_URL, else a local server
TOKEN = None    # None -> QBS_MCP_TOKEN

qbs = connect(URL, token=TOKEN)
qbs

## What the server offers
Each tool's description is what a model routes by — worth reading once.

In [ ]:
for t in qbs.tools():
    print(f"{t['name']:22} {t['description'].splitlines()[0]}")

## Reports, as text
Every tool is a method. The text carries its own caveats — keep them with the numbers.

In [ ]:
print(qbs.current_picks())

In [ ]:
print(qbs.market_overview(sessions=5))

## The JSON tools, as data
`stock_data` and `market_snapshot` return JSON; `call_json` parses it, so the numbers go straight into pandas. A refusal (a name outside the focus list) comes back as text, never as a dict.

In [ ]:
import pandas as pd

picks = ["MU", "MRVL", "AMD", "NVDA", "AVGO"]
rows = []
for t in picks:
    ctx = qbs.call_json("stock_data", ticker=t)
    if not isinstance(ctx, dict):
        print(t, "->", ctx)
        continue
    rows.append({
        "ticker": t,
        "mom_rank": ctx["normal_momentum"]["rank"],
        "mom_held": ctx["normal_momentum"]["currently_held"],
        "resid_rank": ctx["residual_momentum"]["rank"],
        "resid_held": ctx["residual_momentum"]["currently_held"],
    })
pd.DataFrame(rows).set_index("ticker")

In [ ]:
mkt = qbs.call_json("market_snapshot")
pd.Series(mkt["breadth"])

## Backtest statistics
The first call runs the backtest on the server (a few seconds); later calls reuse it.

In [ ]:
print(qbs.strategy_performance())

## Your own agent over the same tools

`langchain_tools()` turns the server's tools into LangChain tools. This is exactly what
the Gemini analyst does when `QBS_MCP_URL` is set — here with a question of your own.

In [ ]:
from qbs.agent.analyst import analyse

tools = qbs.langchain_tools(["current_picks", "name_momentum", "price_action",
                             "market_overview", "fundamentals"])
answer = analyse("Which of today's momentum picks looks most extended, and why?",
                 tools=tools)
print(answer.text)
print("\nTools called:", answer.tools_used)

## After the data is refreshed
When the dashboard or `run_backtest.py --refresh` has updated the cache on the server's machine:

In [ ]:
print(qbs.reload_data())

In [ ]:
qbs.close()